# VidalBralo

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.VidalBralo)

class VidalBralo(pyagingModel):
    def __init__(self):
        super().__init__()

    def preprocess(self, x):
        if self.reference_values is None:
            return x
        if isinstance(self.reference_values, torch.Tensor):
            reference = self.reference_values.to(device=x.device, dtype=x.dtype)
        else:
            reference = torch.tensor(self.reference_values, device=x.device, dtype=x.dtype)
        return torch.where(torch.isnan(x), reference, x)

    def postprocess(self, x):
        return x



In [3]:
model = pya.models.VidalBralo()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "vidalbralo"
model.metadata["data_type"] = "DNA methylation"  # Paper: DNA methylation age measures
model.metadata["species"] = "Homo sapiens"  # Paper: 390 healthy Caucasian donors
model.metadata["year"] = 2016
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Vidal-Bralo, Laura, Yolanda Lopez-Golan, and Antonio Gonzalez. \"Simplified assay for epigenetic age estimation in whole blood of adults.\" Frontiers in Genetics 7 (2016): 126."
model.metadata["doi"] = "https://doi.org/10.3389/fgene.2016.00126"
model.metadata["notes"] = "Eight-CpG whole-blood chronological-age estimator selected by forward stepwise regression in 390 adults and calibrated by multiple linear regression; CpGs were chosen for compatibility with a single multiplex MS-SNuPE assay."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: whole blood (WB)
model.metadata["predicts"] = ["chronological age"]  # Paper: estimate age from blood DNA
model.metadata["training_target"] = ["chronological age"]  # Paper: forward stepwise linear regression with age
model.metadata["unit"] = ["years"]  # Paper: Age range; MAD ... years
model.metadata["model_type"] = "linear regression"  # Paper: multiple linear regression parameters of the 8 CpG DmAM
model.metadata["platform"] = ["Illumina 27K"]  # Paper: training ... obtained with the Illumina Human Methylation 27K BeadChip
model.metadata["population"] = "adults"  # Paper: 390 healthy subjects older than 20 years
model.metadata["journal"] = "Frontiers in Genetics"
model.metadata["last_author"] = "Antonio Gonzalez"
model.metadata["n_features"] = 8
model.metadata["citations"] = 145
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
# The 8-CpG DmAM coefficients are given directly in Table 2 of
# Vidal-Bralo et al. 2016 (Front. Genet. 7:126); they are defined inline below,
# so no external download is required.

## Load features

In [6]:
# Multiple linear regression parameters of the 8 CpG DmAM (Vidal-Bralo et al. 2016, Table 2)
coef_df = pd.DataFrame({
    'CpG': ['cg16386080', 'cg24768561', 'cg19761273', 'cg25809905',
            'cg09809672', 'cg02228185', 'cg17471102', 'cg10917602'],
    'coefficient': [59.5, 33.9, -44.0, -19.7, -22.8, -16.8, -17.7, -11.4],
})
model.features = coef_df['CpG'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['coefficient'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([84.7]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Vidal-Bralo, Laura, Yolanda Lopez-Golan, and Antonio Gonzalez. '
             '"Simplified assay for epigenetic age estimation in whole blood '
             'of adults." Frontiers in genetics 7 (2016): 126.',
 'clock_name': 'vidalbralo',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.3389/fgene.2016.00126',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2016}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg16386080',
 'cg24768561',
 'cg19761273',
 'cg25809905',
 'cg09809672',
 'cg02228185',
 'cg17471102',
 'cg10917602']
base_model_features: None

%==================================== Model Details ====================================%
Model Structure:

base_mode

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)